In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import yaml as yml
import pycountry
from plotly import express as px
import json
from plotly.subplots import make_subplots
from plotly import graph_objects as go
from IPython.display import display, Markdown
from matplotlib import pyplot as plt

from emu_renewal.inputs import BASE_PATH, DATA_PATH, get_standard_priors, get_first_date_above_cov, get_country_vacc_data
from emu_renewal.inputs import get_worldbank_national_pop, get_country_mobility, get_standard_targets, get_euro_var_inputs, get_row_proportions, get_all_var_data, get_country_var_data

In [ ]:
# Standard inputs
analysis_start = datetime(2020, 9, 1)
analysis_end = datetime(2021, 12, 31)
init_duration = 50
var_target_start_date = datetime(2021, 1, 1)
var_target_end_date = datetime(2024, 4, 1)
seed_duration = 10
date_format = "%d/%m/%Y"

In [ ]:
initial_countries = json.load(open(DATA_PATH / f"config/countries.json", "r"))
all_countries = initial_countries["admissions"] + initial_countries["occupancy"]

In [ ]:
cases_fig, cases_axes = plt.subplots(5, 4, figsize=[10, 11], sharex=True)
cases_fig.suptitle("cases")
hosp_fig, hosp_axes = plt.subplots(5, 4, figsize=[10, 11], sharex=True)
hosp_fig.suptitle("hospitalisation")
deaths_fig, deaths_axes = plt.subplots(5, 4, figsize=[10, 11], sharex=True)
deaths_fig.suptitle("deaths")
sero_fig, sero_axes = plt.subplots(5, 4, figsize=[10, 11], sharex=True)
sero_fig.suptitle("seroprevalence")
mob_fig, mob_axes = plt.subplots(5, 4, figsize=[10, 11], sharex=True)
mob_fig.suptitle("mobility")
var_fig, var_axes = plt.subplots(5, 4, figsize=[10, 11], sharex=True)
var_fig.suptitle("variants")
path = BASE_PATH / "preview_plots"
text = ""
flat_cases_axes = cases_axes.ravel()
flat_hosp_axes = hosp_axes.ravel()
flat_deaths_axes = deaths_axes.ravel()
flat_sero_axes = sero_axes.ravel()
flat_mob_axes = mob_axes.ravel()
flat_var_axes = var_axes.ravel()
var_data = get_all_var_data()
for c, country in enumerate(all_countries):
    iso3 = pycountry.countries.lookup(country).alpha_3
    country_name = pycountry.countries.lookup(country).name
    pop = get_worldbank_national_pop(iso3)
    hosp_indicator, hosp_colour = ("Weekly new hospital admissions", "black") if country in initial_countries["admissions"] else ("Daily hospital occupancy", "red")
    cases_target, hosp_target, deaths_target, seroprev_target, init_data = get_standard_targets(
        country, analysis_start, analysis_end, init_duration, hosp_indicator,
    )
    display(Markdown(f"### {country_name}\n\nISO3: {iso3}\n"))
    display(Markdown(f"population: {round(pop / 1e6, 3)} million\n"))
    coverage_threshold = 0.25
    if iso3 == "CHE":
        display(Markdown("Switzerland doesn't seem to have vaccination coverage data available in OWID"))
        coverage_threshold_time = datetime(2021, 3, 26)
    else:
        coverage_threshold_time = get_first_date_above_cov(iso3, coverage_threshold)
    display(Markdown(f"date that vaccination coverage passes {coverage_threshold * 100}%: {datetime.strftime(coverage_threshold_time, '%d/%m/%Y')}"))
    case_ax = flat_cases_axes[c]
    case_ax.plot(cases_target.index, cases_target, linewidth=0, marker=".")
    case_ax.set_title(country_name)
    plt.setp(case_ax.xaxis.get_majorticklabels(), rotation=70)
    hosp_ax = flat_hosp_axes[c]
    hosp_ax.plot(hosp_target.index, hosp_target, linewidth=0, marker=".", color=hosp_colour)
    hosp_ax.set_title(country_name)
    plt.setp(hosp_ax.xaxis.get_majorticklabels(), rotation=70)
    death_ax = flat_deaths_axes[c]
    death_ax.plot(deaths_target.index, deaths_target, linewidth=0, marker=".")
    death_ax.set_title(country_name)
    plt.setp(death_ax.xaxis.get_majorticklabels(), rotation=70)
    sero_ax = flat_sero_axes[c]
    sero_ax.plot(seroprev_target.index, seroprev_target, linewidth=0, marker=".")
    sero_ax.set_title(country_name)
    plt.setp(sero_ax.xaxis.get_majorticklabels(), rotation=70)
    mob_ax = flat_mob_axes[c]
    mob_ax.set_title(country_name)
    plt.setp(mob_ax.xaxis.get_majorticklabels(), rotation=70)
    mobility = get_country_mobility(iso3)
    mob_ax.plot(mobility.index, mobility)
    var_country_name = pycountry.countries.lookup(iso3).official_name if iso3 in ["CZE"] else pycountry.countries.lookup(iso3).name
    country_data = get_country_var_data(var_data, var_country_name)
    var_ax = flat_var_axes[c]
    var_ax.set_title(country_name)
    if country_name != "Serbia":
        get_row_proportions(country_data).plot.area(ax=var_ax, legend=False)
    plt.setp(var_ax.xaxis.get_majorticklabels(), rotation=70)

cases_fig.tight_layout()
cases_fig.savefig("cases_fig.svg")
plt.close()
hosp_fig.tight_layout()
hosp_fig.savefig("hosp_fig.svg")
plt.close()
deaths_fig.tight_layout()
deaths_fig.savefig("deaths_fig.svg")
plt.close()
sero_fig.tight_layout()
sero_fig.savefig("sero_fig.svg")
plt.close()
mob_fig.tight_layout()
mob_fig.savefig("mob_fig.svg")
plt.close()
var_fig.tight_layout()
var_fig.savefig("var_fig.svg")
plt.close()

![Time series of case notifications.](cases_fig.svg)

![Time series of hospitalisation indicator. Admissions, black; occupancy, red.](hosp_fig.svg)

![Time series of deaths.](deaths_fig.svg)

![Seroprevalence estimates over time.](sero_fig.svg)

![Mobility input estimates.](mob_fig.svg)

![Variant proportions.](var_fig.svg)